# 01 Data Ingestion and EDA

This notebook fetches monthly petroleum prices from the EIA API v2, mocks a Thailand DOEB-style monthly series, merges the two datasets on a monthly index, and plots a first comparison view.

In [ ]:
from __future__ import annotations

import time
import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (14, 6)

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
@dataclass
class EIADataFetcher:
    """Thin client for the EIA API v2 with pagination and basic rate-limit retries."""

    api_key: str
    base_url: str = "https://api.eia.gov/v2"
    timeout: int = 30
    max_retries: int = 5
    backoff_factor: float = 1.5
    session: requests.Session = field(default_factory=requests.Session)

    def _request(self, endpoint: str, params: dict[str, Any]) -> dict[str, Any]:
        url = f"{self.base_url.rstrip('/')}/{endpoint.lstrip('/')}"
        request_params = {"api_key": self.api_key, **params}

        for attempt in range(self.max_retries):
            response = self.session.get(url, params=request_params, timeout=self.timeout)

            if response.status_code == 429:
                wait_seconds = self.backoff_factor ** attempt
                print(f"Rate limited by EIA API. Retrying in {wait_seconds:.1f}s...")
                time.sleep(wait_seconds)
                continue

            response.raise_for_status()
            payload = response.json()

            if "response" not in payload:
                raise ValueError(f"Unexpected EIA payload: {payload}")

            return payload

        raise RuntimeError("EIA API retries exhausted after repeated rate-limit responses.")

    def fetch_monthly_petroleum_prices(
        self,
        *,
        product: str = "EPM0",
        duoarea: str = "NUS",
        start: str = "2018-01",
        end: str | None = None,
        page_size: int = 5000,
    ) -> pd.DataFrame:
        """
        Fetch monthly U.S. petroleum price data from EIA API v2.

        Endpoint family used here:
        petroleum/pri/spt/data

        product='EPM0' and duoarea='NUS' are common defaults for a national-level motor gasoline price series.
        Adjust facets if your hackathon team settles on a different petroleum series.
        """

        endpoint = "petroleum/pri/spt/data/"
        offset = 0
        all_rows: list[dict[str, Any]] = []

        while True:
            params: dict[str, Any] = {
                "frequency": "monthly",
                "data[0]": "value",
                "facets[product][]": product,
                "facets[duoarea][]": duoarea,
                "sort[0][column]": "period",
                "sort[0][direction]": "asc",
                "offset": offset,
                "length": page_size,
                "start": start,
            }
            if end is not None:
                params["end"] = end

            payload = self._request(endpoint, params)
            response_block = payload["response"]
            rows = response_block.get("data", [])

            if not rows:
                break

            all_rows.extend(rows)
            total = int(response_block.get("total", len(all_rows)))
            offset += len(rows)

            print(f"Fetched {len(all_rows):,} / {total:,} EIA rows")

            if offset >= total or len(rows) < page_size:
                break

        if not all_rows:
            raise ValueError("EIA API returned no rows for the requested facet filters.")

        df = pd.DataFrame(all_rows)
        df["Date"] = pd.to_datetime(df["period"], format="%Y-%m")
        df["EIA_Price"] = pd.to_numeric(df["value"], errors="coerce")

        keep_columns = [
            "Date",
            "EIA_Price",
            "product",
            "duoarea",
            "units",
            "series-description",
        ]
        available_columns = [col for col in keep_columns if col in df.columns]
        result = (
            df[available_columns]
            .dropna(subset=["EIA_Price"])
            .sort_values("Date")
            .drop_duplicates(subset=["Date"])
            .set_index("Date")
            .asfreq("MS")
        )
        return result

In [ ]:
def load_env_file(env_path: str | Path = "../.env") -> None:
    env_path = Path(env_path)
    if not env_path.exists():
        return

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


load_env_file()
EIA_API_KEY = os.getenv("EIA_API_KEY")
if not EIA_API_KEY:
    raise ValueError("Missing EIA_API_KEY. Add it to ../.env before running this notebook.")

fetcher = EIADataFetcher(api_key=EIA_API_KEY)

# If the chosen facet stops working, inspect the payload and switch to a valid product/duoarea pair.
eia_df = fetcher.fetch_monthly_petroleum_prices(
    product="EPM0",
    duoarea="NUS",
    start="2018-01",
)

eia_df.head()

In [ ]:
def build_mock_doeb_series(date_index: pd.DatetimeIndex, seed: int = 42) -> pd.DataFrame:
    """
    Mock a DOEB-style monthly volume series aligned to the EIA date index.

    The synthetic process intentionally includes:
    - trend: long-run growth in local energy demand
    - seasonality: recurring monthly usage pattern
    - lagged relationship: partial dependence on previous month's global price
    - noise: real-world measurement and behavioral variation
    """

    rng = np.random.default_rng(seed)
    n_periods = len(date_index)
    month_number = np.arange(n_periods)

    trend = np.linspace(120, 175, n_periods)
    seasonality = 12 * np.sin(2 * np.pi * month_number / 12)
    lagged_price_signal = eia_df["EIA_Price"].shift(1).bfill().to_numpy() * 1.8
    noise = rng.normal(loc=0.0, scale=4.0, size=n_periods)

    doeb_volume = trend + seasonality + lagged_price_signal + noise

    return pd.DataFrame(
        {"DOEB_Volume": doeb_volume},
        index=date_index,
    )


doeb_df = build_mock_doeb_series(eia_df.index)
doeb_df.head()

In [ ]:
merged_df = (
    eia_df[["EIA_Price"]]
    .join(doeb_df, how="inner")
    .sort_index()
)

merged_df.index.name = "Date"
merged_df.to_csv(ARTIFACT_DIR / "merged_monthly_prices.csv")

print(merged_df.shape)
merged_df.head()

In [ ]:
fig, ax = plt.subplots()

ax.plot(merged_df.index, merged_df["EIA_Price"], label="EIA Price", linewidth=2)
ax.set_ylabel("EIA Price", color="tab:blue")
ax.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax.twinx()
ax2.plot(merged_df.index, merged_df["DOEB_Volume"], label="DOEB Volume", linewidth=2, color="tab:orange")
ax2.set_ylabel("DOEB Volume", color="tab:orange")
ax2.tick_params(axis="y", labelcolor="tab:orange")

ax.set_title("Monthly Comparison: EIA Petroleum Price vs Mock DOEB Volume")
ax.set_xlabel("Month")
fig.tight_layout()
plt.show()